In [1]:
pip install pandas openpyxl

In [2]:
pip install rpy2

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from pathlib import Path

# 1 TRATAMENTO DE DADOS & AMOSTRAGEM

### 1.1 Função DB da O*NET

O script a seguir automatiza a importação de múltiplas versões das bases de dados da O*NET e OESM. Inicialmente, define-se o diretório principal contendo os arquivos e gera-se automaticamente uma lista com as versões disponíveis das bases.

Em seguida, a função carregar_bases() percorre cada versão, constrói dinamicamente o caminho dos arquivos e realiza a leitura das planilhas Excel utilizando a biblioteca pandas. As bases importadas são armazenadas em um dicionário com identificação automática das versões, permitindo acesso organizado e padronizado aos dados.

O código também utiliza tratamento de exceções (try/except) para identificar arquivos ausentes sem interromper a execução do processo. Essa abordagem aumenta a reprodutibilidade, reduz erros de importação manual e facilita o gerenciamento de diferentes versões das bases de dados.

In [4]:
base_path = Path(r'C:\Users\Maxwell\Documents\mestrado\dados')

# versões O*NET
versoes_onet = [
    f"{ano}_{sub}"
    for ano in range(21, 31)
    for sub in range(4)
]

versoes_oesm = range(16, 26)


def carregar_onet(nome_arquivo, prefixo):

    dfs = {}

    for versao in versoes_onet:

        pasta = base_path / "onet" / f"db_{versao}_excel" / f"db_{versao}_excel"
        caminho = pasta / nome_arquivo

        try:
            chave = f'{prefixo}{versao.replace("_","")}'
            dfs[chave] = pd.read_excel(caminho)

            print(f'✔ {chave}')

        except FileNotFoundError:
            print(f'✘ Não encontrado: {versao}')

    return dfs


def carregar_oesm(prefixo="oesm"):

    dfs = {}

    for ano in versoes_oesm:

        pasta = base_path / "oesm" / f"oesm{ano}nat" / f"oesm{ano}nat"
        caminho = pasta / f"national_M20{ano}_dl.xlsx"

        try:
            chave = f"{prefixo}{ano}"
            dfs[chave] = pd.read_excel(caminho)

            print(f'✔ {chave}')

        except FileNotFoundError:
            print(f'✘ Não encontrado: {ano}')

    return dfs

In [5]:
oesm = carregar_oesm()

✔ oesm16
✔ oesm17
✔ oesm18
✔ oesm19
✔ oesm20
✔ oesm21
✔ oesm22
✔ oesm23
✔ oesm24
✔ oesm25


In [6]:
occupation_dfs = carregar_onet('Occupation Data.xlsx', 'od')

✘ Não encontrado: 21_0
✘ Não encontrado: 21_1
✔ od212
✔ od213
✔ od220
✔ od221
✔ od222
✔ od223
✔ od230
✔ od231
✔ od232
✔ od233
✔ od240
✔ od241
✔ od242
✔ od243
✔ od250
✔ od251
✔ od252
✔ od253
✔ od260
✔ od261
✔ od262
✔ od263
✔ od270
✔ od271
✔ od272
✔ od273
✔ od280
✔ od281
✔ od282
✔ od283
✔ od290
✔ od291
✔ od292
✔ od293
✔ od300
✔ od301
✔ od302
✔ od303


### 1.2 Mesclar OESM com O*NET

O script realizou a integração entre as bases da ONET (“Occupation Data”) e da OEWS/OESM, utilizando o código ocupacional SOC como chave de correspondência. Para isso, os códigos da ONET presentes na coluna "O*NET-SOC Code" foram padronizados para o formato utilizado pela OESM, removendo o sufixo decimal .00 (ex.: 11-1011.00 → 11-1011). Em seguida, cada versão da O*NET foi associada ao respectivo ano da OESM por meio de um mapeamento temporal entre as bases.

Após a padronização, foi realizado um merge entre as bases utilizando as colunas "SOC_MATCH" (O*NET) e "OCC_CODE" (OESM). Da OESM foram incorporadas as variáveis "OCC_TITLE", "TOT_EMP", "H_MEAN" e "A_MEAN", correspondentes respectivamente ao título ocupacional, total de empregados, salário médio por hora e salário médio anual.

As ocupações sem correspondência na OESM foram identificadas e armazenadas em um relatório específico (relatorio_sem_match), sendo posteriormente removidas da base final. O resultado consolidado foi armazenado no objeto occupation_dfs_merge, contendo apenas ocupações com correspondência válida entre as duas bases.

Os resultados indicaram elevada compatibilidade entre as classificações ocupacionais da ONET e da OESM para o período 2020/21 - 2025/26. Nas versões analisadas, entre 818 e 819 ocupações foram mantidas após a integração, enquanto entre 197 e 198 ocupações foram removidas por ausência de correspondência na OESM. Essas diferenças decorrem principalmente de ocupações específicas da ONET que não possuem equivalência direta nas tabelas salariais da OESM.

Já o periodo entre 2017 e 2020 apresentam maior variabilidade. Nas versões analisadas, entre 707 e 797 ocupações foram mantidas após a integração, enquanto entre 313 e 407 ocupações foram removidas por ausência de correspondência na OESM. Essas diferenças decorrem principalmente de ocupações específicas da ONET que não possuem equivalência direta nas tabelas salariais da OESM.

In [7]:
# ---------------------------------------------
# RELAÇÃO ENTRE O*NET E OESM
# ---------------------------------------------

mapa_anos = {}

# 2017
for i in range(212, 222):
    mapa_anos[f"od{i}"] = ("oesm17", 2017)

# 2018
for i in range(222, 232):
    mapa_anos[f"od{i}"] = ("oesm18", 2018)

# 2019
for i in range(232, 242):
    mapa_anos[f"od{i}"] = ("oesm19", 2019)

# 2020
for i in range(242, 252):
    mapa_anos[f"od{i}"] = ("oesm20", 2020)

# 2021
for i in range(252, 262):
    mapa_anos[f"od{i}"] = ("oesm21", 2021)

# 2022
for i in range(262, 272):
    mapa_anos[f"od{i}"] = ("oesm22", 2022)

# 2023
for i in range(272, 282):
    mapa_anos[f"od{i}"] = ("oesm23", 2023)

# 2024
for i in range(282, 292):
    mapa_anos[f"od{i}"] = ("oesm24", 2024)

# 2025
for i in range(292, 302):
    mapa_anos[f"od{i}"] = ("oesm25", 2025)

# 2025
for i in range(302, 310):
    mapa_anos[f"od{i}"] = ("oesm25", 2025)

In [8]:
# ---------------------------------------------
# FUNÇÃO PARA INCLUIR DADOS OESM NAS od
# ---------------------------------------------

def integrar_oesm_od(occupation_dfs, oesm):

    relatorio_sem_match = {}
    occupation_dfs_merge = {}
    occupation_dfs_completo = {}

    for chave_od, df_od in occupation_dfs.items():

        print(f"\nProcessando {chave_od}")

        # identifica base OESM e ano
        info = mapa_anos.get(chave_od)

        if info is None:
            print(f"✘ Sem mapeamento para {chave_od}")
            continue

        chave_oesm, ano = info

        df_oesm = oesm[chave_oesm].copy()

        # ---------------------------------------------
        # padroniza códigos SOC
        # ---------------------------------------------
        df_od = df_od.copy()

        df_od["SOC_MATCH"] = (
            df_od["O*NET-SOC Code"]
            .astype(str)
            .str.replace(r"\.00$", "", regex=True)
        )

        # ---------------------------------------------
        # seleciona colunas OESM
        # ---------------------------------------------
        df_oesm.columns = df_oesm.columns.str.upper()

        df_oesm = df_oesm[
            ["OCC_CODE", "OCC_TITLE", "TOT_EMP", "H_MEAN", "A_MEAN"]
        ].copy()

        # ---------------------------------------------
        # merge
        # ---------------------------------------------
        df_merge = df_od.merge(
            df_oesm,
            left_on="SOC_MATCH",
            right_on="OCC_CODE",
            how="left"
        )

        # ---------------------------------------------
        # adiciona YEAR
        # ---------------------------------------------
        df_merge["YEAR"] = ano

        # ---------------------------------------------
        # cria coluna indicando match
        # ---------------------------------------------
        df_merge["MATCH_OESM"] = (
            df_merge["TOT_EMP"].notna()
        )

        # dataframe completo
        occupation_dfs_completo[chave_od] = df_merge.copy()

        # relatório sem match
        sem_match = df_merge[
            df_merge["MATCH_OESM"] == False
        ].copy()

        relatorio_sem_match[chave_od] = sem_match[
            ["O*NET-SOC Code", "Title"]
        ].drop_duplicates()

        # mantém apenas matches
        df_match = df_merge[
            df_merge["MATCH_OESM"] == True
        ].copy()

        print(
            f"✔ Mantidos: {len(df_match)} | "
            f"✘ Removidos: {len(relatorio_sem_match[chave_od])}"
        )

        # remove colunas auxiliares
        df_match = df_match.drop(
            columns=["SOC_MATCH", "OCC_CODE"],
            errors="ignore"
        )

        occupation_dfs_merge[chave_od] = df_match

    return (
        occupation_dfs_merge,
        relatorio_sem_match,
        occupation_dfs_completo
    )

In [9]:
(
    occupation_dfs_merge,
    relatorio_sem_match,
    occupation_dfs_completo
) = integrar_oesm_od(occupation_dfs, oesm)


Processando od212
✔ Mantidos: 797 | ✘ Removidos: 313

Processando od213
✔ Mantidos: 797 | ✘ Removidos: 313

Processando od220
✔ Mantidos: 797 | ✘ Removidos: 313

Processando od221
✔ Mantidos: 797 | ✘ Removidos: 313

Processando od222
✔ Mantidos: 796 | ✘ Removidos: 314

Processando od223
✔ Mantidos: 796 | ✘ Removidos: 314

Processando od230
✔ Mantidos: 796 | ✘ Removidos: 314

Processando od231
✔ Mantidos: 796 | ✘ Removidos: 314

Processando od232
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od233
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od240
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od241
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od242
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od243
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od250
✔ Mantidos: 703 | ✘ Removidos: 407

Processando od251
✔ Mantidos: 747 | ✘ Removidos: 269

Processando od252
✔ Mantidos: 819 | ✘ Removidos: 197

Processando od253
✔ Mantidos: 819 | ✘ Removidos: 197

Processando od260
✔ Mantido

### 1.3 Validação da Mesclagem O*NET-OESM

Após a integração entre as bases da ONET e da OESM, foi realizado um procedimento adicional de validação para verificar a consistência dos títulos ocupacionais associados aos códigos SOC correspondentes. Para isso, foram comparadas as colunas "Title" da ONET e "OCC_TITLE" da OESM nas bases integradas (occupation_dfs_merge).

A comparação foi feita após a padronização textual dos títulos, convertendo os caracteres para letras minúsculas (lower()), removendo espaços excedentes (strip()) e transformando os valores em formato textual (astype(str)), reduzindo diferenças decorrentes apenas de formatação. Em seguida, foi criada a variável lógica "MATCH_TITLE", indicando se os títulos ocupacionais eram idênticos entre as duas bases.

Os resultados demonstraram correspondência total entre os títulos ocupacionais da O*NET e da OESM, coreespondente a 2020/21 - 2025/26, em todas as versões analisadas. Em todas as bases processadas, 100% das ocupações integradas apresentaram equivalência textual entre "Title" e "OCC_TITLE", sem ocorrência de divergências. Esse resultado confirma a consistência do procedimento de pareamento realizado a partir dos códigos SOC e reforça a compatibilidade estrutural entre as classificações ocupacionais utilizadas pelas duas bases de dados para este periodo.

No entanto no periodo 2017 - 2020, foi observada divergências entre "Title" (O*NET) e "OCC_TITLE" (OESM). Entretanto a análise qualitativa, a partir do relatório das diferenças (Excel), indica que, em praticamente todos os casos, as diferenças observadas não representam ocupações distintas, mas sim atualizações de nomenclatura, padronizações terminológicas ou mudanças de escopo descritivo realizadas pelo sistema SOC/OESM ao longo do tempo.

As divergências identificadas podem ser agrupadas em alguns padrões principais:

substituição de termos equivalentes, como "Gaming" → "Gambling":
Gaming Managers → Gambling Managers |
Gaming Dealers → Gambling Dealers

atualização para nomenclaturas mais amplas ou recentes:
Health Educators → Health Education Specialists |
Self-Enrichment Education Teachers → Self-Enrichment Teachers |
Parking Lot Attendants → Parking Attendants

inclusão explícita de subáreas ou funções:
Radiologic Technologists → Radiologic Technologists and Technicians |
Audio and Video Equipment Technicians → Audio and Video Technicians |
Elevator Installers and Repairers → Elevator and Escalator Installers and Repairers

mudança para terminologias padronizadas pelo SOC:
Police, Fire, and Ambulance Dispatchers → Public Safety Telecommunicators |
Legal Secretaries → Legal Secretaries and Administrative Assistants |
Medical Secretaries → Medical Secretaries and Administrative Assistants

ampliação do escopo ocupacional:
Shipping, Receiving, and Traffic Clerks → Shipping, Receiving, and Inventory Clerks |
Tile and Marble Setters → Tile and Stone Setters

Além disso, parte das diferenças decorre da atualização estrutural do sistema SOC 2018/2020, especialmente em ocupações técnicas e tecnológicas, nas quais a expressão "Technologists and Technicians" passou a substituir antigas denominações apenas com "Technicians":

Civil Engineering Technicians → Civil Engineering Technologists and Technicians
Mechanical Engineering Technicians → Mechanical Engineering Technologists and Technicians

Entre todas as divergências observadas, não foram identificados casos evidentes em que o mesmo código SOC estivesse associado a ocupações substantivamente diferentes entre O*NET e OESM. As diferenças encontradas refletem predominantemente revisões terminológicas, harmonizações institucionais e atualizações oficiais da classificação ocupacional, preservando a equivalência ocupacional dos códigos analisados.

In [33]:
# ---------------------------------------------
# RELATÓRIO DE MATCH DOS TÍTULOS
# OESM: OCC_TITLE
# O*NET: Title
# ---------------------------------------------

relatorio_match_titulos = {}

for chave, df in occupation_dfs_merge.items():

    relatorio = df[
        [
            "O*NET-SOC Code",
            "Title",
            "OCC_TITLE"
        ]
    ].copy()

    # verifica se os títulos são iguais
    relatorio["MATCH_TITLE"] = (
        relatorio["Title"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        relatorio["OCC_TITLE"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    relatorio_match_titulos[chave] = relatorio

    total = len(relatorio)
    matches = relatorio["MATCH_TITLE"].sum()
    nao_matches = total - matches

    print(
        f"{chave} | "
        f"✔ Matches: {matches} | "
        f"✘ Diferenças: {nao_matches}"
    )

od212 | ✔ Matches: 796 | ✘ Diferenças: 1
od213 | ✔ Matches: 796 | ✘ Diferenças: 1
od220 | ✔ Matches: 796 | ✘ Diferenças: 1
od221 | ✔ Matches: 796 | ✘ Diferenças: 1
od222 | ✔ Matches: 795 | ✘ Diferenças: 1
od223 | ✔ Matches: 795 | ✘ Diferenças: 1
od230 | ✔ Matches: 795 | ✘ Diferenças: 1
od231 | ✔ Matches: 795 | ✘ Diferenças: 1
od232 | ✔ Matches: 653 | ✘ Diferenças: 50
od233 | ✔ Matches: 653 | ✘ Diferenças: 50
od240 | ✔ Matches: 653 | ✘ Diferenças: 50
od241 | ✔ Matches: 653 | ✘ Diferenças: 50
od242 | ✔ Matches: 653 | ✘ Diferenças: 50
od243 | ✔ Matches: 653 | ✘ Diferenças: 50
od250 | ✔ Matches: 653 | ✘ Diferenças: 50
od251 | ✔ Matches: 747 | ✘ Diferenças: 0
od252 | ✔ Matches: 819 | ✘ Diferenças: 0
od253 | ✔ Matches: 819 | ✘ Diferenças: 0
od260 | ✔ Matches: 819 | ✘ Diferenças: 0
od261 | ✔ Matches: 819 | ✘ Diferenças: 0
od262 | ✔ Matches: 818 | ✘ Diferenças: 0
od263 | ✔ Matches: 818 | ✘ Diferenças: 0
od270 | ✔ Matches: 818 | ✘ Diferenças: 0
od271 | ✔ Matches: 818 | ✘ Diferenças: 0
od272 | ✔

### 1.4 Validação das Versões do O*NET

Foi realizado um procedimento de comparação longitudinal entre as diferentes versões da base “Occupation Data” da ONET já integradas à OESM (occupation_dfs_merge). O objetivo foi identificar alterações estruturais nas ocupações ao longo das atualizações da ONET, verificando a permanência, inclusão, exclusão ou alteração dos títulos ocupacionais associados aos códigos SOC após o processo de pareamento com a OESM.

Para cada par consecutivo de versões, foram comparadas as colunas "O*NET-SOC Code" e "Title". A análise classificou cada ocupação em quatro categorias: (i) "mantido" quando o código SOC permaneceu presente com o mesmo título; (ii) "novo" quando o código apareceu apenas na versão mais recente; (iii) "excluido" quando o código existia apenas na versão anterior; e (iv) "titulo_atualizado" quando o código permaneceu, mas o título ocupacional foi alterado.

Os resultados indicaram elevada estabilidade estrutural entre as versões analisadas, embora tenham sido observadas diferenças relevantes em períodos específicos da série histórica. Entre od212 e od231, a amostra integrada permaneceu praticamente estável, com manutenção de aproximadamente 796–797 ocupações por versão e apenas uma exclusão pontual entre od221 e od222.

A principal alteração ocorreu entre od231 e od232, quando 93 ocupações deixaram de compor a amostra integrada, reduzindo o total de ocupações mantidas de 796 para 703. Como não foram observadas inclusões de novas ocupações nem atualizações de títulos nesse intervalo, esse resultado sugere uma redução substancial da compatibilidade entre a O*NET e a OESM nesse período específico, provavelmente decorrente de mudanças metodológicas ou estruturais nas tabelas ocupacionais utilizadas para o pareamento.

Entre od232 e od250, a estrutura permaneceu estável, mantendo aproximadamente 703 ocupações constantes ao longo das atualizações consecutivas. Posteriormente, entre od250 e od251, ocorreu a principal mudança estrutural da série analisada. Nesse intervalo foram identificadas simultaneamente:

51 ocupações com atualização de títulos;
46 novas ocupações;
2 ocupações excluídas;
redução das ocupações mantidas de 703 para 650.

Esse padrão indica uma revisão estrutural significativa da classificação ocupacional da ONET, compatível com mudanças associadas à atualização do sistema SOC/ONET. Diferentemente da quebra observada entre od231 e od232, as alterações entre od250 e od251 refletem predominantemente mudanças internas da própria O*NET, incluindo reclassificações ocupacionais e atualização de nomenclaturas.

Entre od251 e od252, observou-se recuperação parcial da cobertura ocupacional, com inclusão de 72 novas ocupações e aumento do total de ocupações mantidas de 747 para 819. A partir desse ponto, a amostra tornou-se altamente estável, mantendo entre 818 e 819 ocupações ao longo das versões subsequentes.

Após od252, não foram identificadas novas alterações de títulos ocupacionais ("titulo_atualizado" = 0), indicando elevada consistência nominal entre as versões recentes da O*NET. As únicas mudanças observadas corresponderam a eventos pontuais de inclusão ou exclusão de ocupações específicas:

exclusão de "29-1024.00 – Prosthodontists" entre od261 e od262, posteriormente reincluída entre od271 e od272;
exclusão de "11-1031.00 – Legislators" entre od291 e od292.

De forma geral, os resultados sugerem que as diferenças observadas na amostra integrada decorreram de dois processos distintos: (i) mudanças de compatibilidade entre ONET e OESM, particularmente evidentes entre od231 e od232; e (ii) revisões estruturais da própria ONET, especialmente concentradas na transição entre od250 e od251. Após essa reestruturação, a amostra apresentou elevada estabilidade longitudinal e forte consistência ocupacional entre as bases integradas.

In [34]:
# ---------------------------------------------
# FUNÇÃO GERAL DE RELATÓRIO ENTRE VERSÕES
# ---------------------------------------------

def relatorio_alteracoes(df_dict, nome_relatorio):

    relatorio = []

    # ordena versões
    chaves = sorted(df_dict.keys())

    for i in range(len(chaves) - 1):

        atual = chaves[i]
        prox = chaves[i + 1]

        print(f"\n{nome_relatorio}: {atual} -> {prox}")

        # ---------------------------------------------
        # seleciona colunas
        # ---------------------------------------------
        df_atual = df_dict[atual][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        df_prox = df_dict[prox][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        # renomeia títulos
        df_atual = df_atual.rename(
            columns={"Title": "Title_atual"}
        )

        df_prox = df_prox.rename(
            columns={"Title": "Title_prox"}
        )

        # ---------------------------------------------
        # comparação
        # ---------------------------------------------
        comp = df_atual.merge(
            df_prox,
            on="O*NET-SOC Code",
            how="outer",
            indicator=True
        )

        # ---------------------------------------------
        # status inicial
        # ---------------------------------------------
        comp["STATUS"] = "mantido"

        # excluído
        comp.loc[
            comp["_merge"] == "left_only",
            "STATUS"
        ] = "excluido"

        # novo
        comp.loc[
            comp["_merge"] == "right_only",
            "STATUS"
        ] = "novo"

        # título alterado
        comp.loc[
            (
                comp["_merge"] == "both"
            ) &
            (
                comp["Title_atual"]
                .astype(str)
                .str.strip()
                !=
                comp["Title_prox"]
                .astype(str)
                .str.strip()
            ),
            "STATUS"
        ] = "titulo_atualizado"

        # versões comparadas
        comp["VERSAO_ATUAL"] = atual
        comp["VERSAO_COMPARADA"] = prox

        # adiciona ao relatório final
        relatorio.append(comp)

        # ---------------------------------------------
        # resumo
        # ---------------------------------------------
        resumo = comp["STATUS"].value_counts()

        print(
            f"✔ Mantidos: "
            f"{resumo.get('mantido',0)}"
        )

        print(
            f"✏ Títulos atualizados: "
            f"{resumo.get('titulo_atualizado',0)}"
        )

        print(
            f"➕ Novos: "
            f"{resumo.get('novo',0)}"
        )

        print(
            f"✘ Excluídos: "
            f"{resumo.get('excluido',0)}"
        )

    # concatena tudo
    relatorio = pd.concat(
        relatorio,
        ignore_index=True
    )

    return relatorio

In [35]:
# ---------------------------------------------
# 1. Apenas matches com OESM
# ---------------------------------------------
relatorio_merge = relatorio_alteracoes(
    occupation_dfs_merge,
    "occupation_dfs_merge"
)


occupation_dfs_merge: od212 -> od213
✔ Mantidos: 797
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od213 -> od220
✔ Mantidos: 797
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od220 -> od221
✔ Mantidos: 797
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od221 -> od222
✔ Mantidos: 796
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 1

occupation_dfs_merge: od222 -> od223
✔ Mantidos: 796
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od223 -> od230
✔ Mantidos: 796
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od230 -> od231
✔ Mantidos: 796
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od231 -> od232
✔ Mantidos: 703
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 93

occupation_dfs_merge: od232 -> od233
✔ Mantidos: 703
✏ Títulos atualizados: 0
➕ Novos: 0
✘ Excluídos: 0

occupation_dfs_merge: od233 -> od240
✔ Mantidos: 703


## 1.5 Alterações na Amostragem

A análise longitudinal da integração entre ONET e OESM identificou três tipos principais de mudanças: (i) alterações estruturais da ONET, (ii) problemas de compatibilização com a OESM e (iii) atualizações de títulos ocupacionais.

A estrutura ocupacional apresentou elevada estabilidade ao longo da maior parte da série histórica. Entre od212 e od231, não ocorreram mudanças estruturais na O*NET nem atualizações de títulos ocupacionais. Nesse período, apenas uma alteração pontual de compatibilização foi observada entre od221 e od222, com impacto de 1 ocupação excluída da amostra integrada.

A principal ruptura relacionada ao processo de match ocorreu entre od231 e od232, quando 93 ocupações deixaram de compor a amostra integrada, sem qualquer alteração estrutural da ONET ou atualização de títulos. Isso indica que a perda decorreu predominantemente de limitações de compatibilidade entre as classificações ocupacionais da ONET e da OESM.

Entre od232 e od250, a estrutura da amostra voltou a apresentar estabilidade completa. Nenhuma mudança estrutural, atualização de títulos ou impacto relevante de compatibilização foi identificado nas transições od232→od233, od233→od240, od240→od241, od241→od242, od242→od243 e od243→od250.

A mudança mais significativa de toda a série ocorreu entre od250 e od251. Nesse intervalo foram registradas 430 mudanças estruturais da ONET e 59 atualizações de títulos ocupacionais, sem perdas associadas ao processo de match com a OESM. Essa transição refletiu uma ampla revisão da taxonomia ocupacional da ONET e atualização do sistema SOC, envolvendo simultaneamente exclusão de códigos antigos, criação de novas categorias e modernização terminológica.

Após essa reestruturação, observou-se expansão da cobertura ocupacional entre od251 e od252, com impacto de 72 ocupações associado exclusivamente ao processo de compatibilização com a OESM. Isso sugere que parte das novas ocupações introduzidas pela revisão estrutural passou a encontrar correspondência adequada na OESM apenas nas versões subsequentes.

A partir de od252, a amostra integrada tornou-se novamente altamente estável. Não foram identificadas novas mudanças estruturais da O*NET nem atualizações de títulos ocupacionais até od302. As únicas alterações observadas corresponderam a eventos pontuais de compatibilização com a OESM:

od261→od262: impacto de 1 ocupação excluída;
od271→od272: reinclusão de 1 ocupação;
od291→od292: exclusão de 1 ocupação.

As ocupações excluídas concentraram-se principalmente em categorias antigas, residuais ou altamente específicas, sobretudo nas áreas de TI, engenharia, saúde, ciências, administração e logística. Muitas pertenciam a classificações residuais do SOC, como o grupo 15-1199, posteriormente reorganizado em categorias mais específicas.

As ocupações adicionadas seguiram padrão consistente de maior especialização e digitalização, incluindo expansão de funções ligadas à ciência de dados, analytics, TI, especialidades médicas e supervisão logística.

As 59 atualizações de títulos observadas entre od250 e od251 foram predominantemente terminológicas e institucionais, envolvendo modernização de nomenclaturas, inclusão explícita de “technologists and technicians” e adaptação a novas terminologias tecnológicas e audiovisuais. Em geral, essas alterações não comprometem a comparabilidade longitudinal das ocupações, especialmente em análises agregadas.

As ocupações presentes na O*NET, mas ausentes na OESM, concentraram-se principalmente em cinco grupos: ocupações militares, ocupações extremamente especializadas ou emergentes, categorias residuais (“All Other”), ocupações com baixa frequência amostral e algumas funções governamentais específicas, como legislators. Isso demonstra que as perdas de match não ocorreram aleatoriamente, mas refletem diferenças estruturais entre as bases.

De forma geral, os resultados indicam que as maiores perdas anteriores à od250 decorreram de limitações de compatibilidade entre ONET e OESM, enquanto a transição od250→od251 representou uma mudança estrutural substantiva da própria ONET. Após essa atualização, a amostra integrada apresentou elevada estabilidade longitudinal e forte consistência ocupacional entre as versões analisadas.

In [36]:
# ---------------------------------------------
# IDENTIFICAR A ORIGEM DAS MUDANÇAS
# O*NET vs OESM
# ---------------------------------------------

def diagnostico_mudancas(
    occupation_dfs,
    occupation_dfs_merge
):

    relatorio_causa = []

    chaves = sorted(occupation_dfs.keys())

    for i in range(len(chaves) - 1):

        atual = chaves[i]
        prox = chaves[i + 1]

        print(f"\n{atual} -> {prox}")

        # ---------------------------------------------
        # IGNORA versões ausentes no merge
        # ---------------------------------------------
        if (
            atual not in occupation_dfs_merge
            or
            prox not in occupation_dfs_merge
        ):

            print(
                f"⚠ Comparação ignorada "
                f"(ausente em occupation_dfs_merge)"
            )

            continue

        # ---------------------------------------------
        # BASE ORIGINAL O*NET
        # ---------------------------------------------
        onet_atual = occupation_dfs[atual][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        onet_prox = occupation_dfs[prox][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        # ---------------------------------------------
        # BASE FINAL APÓS MATCH COM OESM
        # ---------------------------------------------
        merge_atual = occupation_dfs_merge[atual][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        merge_prox = occupation_dfs_merge[prox][
            ["O*NET-SOC Code", "Title"]
        ].copy()

        # ---------------------------------------------
        # CÓDIGOS
        # ---------------------------------------------
        set_onet_atual = set(
            onet_atual["O*NET-SOC Code"]
        )

        set_onet_prox = set(
            onet_prox["O*NET-SOC Code"]
        )

        set_merge_atual = set(
            merge_atual["O*NET-SOC Code"]
        )

        set_merge_prox = set(
            merge_prox["O*NET-SOC Code"]
        )

        # ---------------------------------------------
        # DIFERENÇAS O*NET
        # ---------------------------------------------
        excluidos_onet = (
            set_onet_atual - set_onet_prox
        )

        novos_onet = (
            set_onet_prox - set_onet_atual
        )

        # ---------------------------------------------
        # DIFERENÇAS APÓS MATCH
        # ---------------------------------------------
        excluidos_merge = (
            set_merge_atual - set_merge_prox
        )

        novos_merge = (
            set_merge_prox - set_merge_atual
        )

        # ---------------------------------------------
        # IDENTIFICA CAUSA
        # ---------------------------------------------
        mudanca_onet = (
            excluidos_onet.union(novos_onet)
        )

        mudanca_oesm = (
            excluidos_merge.union(novos_merge)
        ) - mudanca_onet

        # ---------------------------------------------
        # TÍTULOS ALTERADOS
        # ---------------------------------------------
        tit_atual = onet_atual.set_index(
            "O*NET-SOC Code"
        )["Title"]

        tit_prox = onet_prox.set_index(
            "O*NET-SOC Code"
        )["Title"]

        codigos_comuns = (
            set_onet_atual.intersection(
                set_onet_prox
            )
        )

        titulos_alterados = []

        for cod in codigos_comuns:

            if (
                str(tit_atual[cod]).strip()
                !=
                str(tit_prox[cod]).strip()
            ):

                titulos_alterados.append(cod)

        # ---------------------------------------------
        # RESUMO
        # ---------------------------------------------
        print(
            f"Mudanças estruturais O*NET: "
            f"{len(mudanca_onet)}"
        )

        print(
            f"Impacto do match OESM: "
            f"{len(mudanca_oesm)}"
        )

        print(
            f"Títulos atualizados: "
            f"{len(titulos_alterados)}"
        )

        # ---------------------------------------------
        # RELATÓRIO DETALHADO
        # ---------------------------------------------
        for cod in mudanca_onet:

            relatorio_causa.append({
                "VERSAO_ATUAL": atual,
                "VERSAO_COMPARADA": prox,
                "O*NET-SOC Code": cod,
                "CAUSA": "mudanca_estrutural_onet"
            })

        for cod in mudanca_oesm:

            relatorio_causa.append({
                "VERSAO_ATUAL": atual,
                "VERSAO_COMPARADA": prox,
                "O*NET-SOC Code": cod,
                "CAUSA": "impacto_match_oesm"
            })

        for cod in titulos_alterados:

            relatorio_causa.append({
                "VERSAO_ATUAL": atual,
                "VERSAO_COMPARADA": prox,
                "O*NET-SOC Code": cod,
                "CAUSA": "titulo_atualizado_onet"
            })

    relatorio_causa = pd.DataFrame(
        relatorio_causa
    )

    return relatorio_causa

In [37]:
relatorio_causa = diagnostico_mudancas(
    occupation_dfs,
    occupation_dfs_merge
)


od212 -> od213
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od213 -> od220
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od220 -> od221
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od221 -> od222
Mudanças estruturais O*NET: 0
Impacto do match OESM: 1
Títulos atualizados: 0

od222 -> od223
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od223 -> od230
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od230 -> od231
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od231 -> od232
Mudanças estruturais O*NET: 0
Impacto do match OESM: 93
Títulos atualizados: 0

od232 -> od233
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od233 -> od240
Mudanças estruturais O*NET: 0
Impacto do match OESM: 0
Títulos atualizados: 0

od240 -> od241
Mudanças estruturais O*NET: 0
Impacto do ma

## 1.6 Seleção das Variáveis

Foram selecionadas como variáveis de análise as tabelas Abilities, Interests Knowledge, Skills, Work Activities Work Styles e Work Values da base de dados da O*NET, contudo, as mesclar essas tabelas: updates assíncronos;
frequência desigual;
construtos com persistência diferente;
painéis ocupacionais irregulares.

Na versão 30.3 da O*NET, a estrutura tradicional do arquivo Skills.xlsx foi modificada, passando as informações de habilidades a serem distribuídas nos arquivos “Transferable Skills” e “Essential Skills”. Assim, realizou-se um procedimento adicional de concatenação dessas duas tabelas, formando uma base unificada equivalente às versões anteriores, posteriormente incorporada ao conjunto de dados de skills para preservar a consistência longitudinal da análise.

In [9]:
abilities_dfs = carregar_onet('Abilities.xlsx', 'ab')
interests_dfs = carregar_onet('Interests.xlsx', 'in')
knowledge_dfs = carregar_onet('Knowledge.xlsx', 'kn')
skills_dfs = carregar_onet('Skills.xlsx', 'sk')
activities_dfs = carregar_onet('Work Activities.xlsx', 'wa')
styles_dfs = carregar_onet('Work Styles.xlsx', 'ws')
values_dfs = carregar_onet('Work Values.xlsx', 'wv')

✘ Não encontrado: 21_0
✘ Não encontrado: 21_1
✔ ab212
✔ ab213
✔ ab220
✔ ab221
✔ ab222
✔ ab223
✔ ab230
✔ ab231
✔ ab232
✔ ab233
✔ ab240
✔ ab241
✔ ab242
✔ ab243
✔ ab250
✔ ab251
✔ ab252
✔ ab253
✔ ab260
✔ ab261
✔ ab262
✔ ab263
✔ ab270
✔ ab271
✔ ab272
✔ ab273
✔ ab280
✔ ab281
✔ ab282
✔ ab283
✔ ab290
✔ ab291
✔ ab292
✔ ab293
✔ ab300
✔ ab301
✔ ab302
✔ ab303
✘ Não encontrado: 21_0
✘ Não encontrado: 21_1
✔ in212
✔ in213
✔ in220
✔ in221
✔ in222
✔ in223
✔ in230
✔ in231
✔ in232
✔ in233
✔ in240
✔ in241
✔ in242
✔ in243
✔ in250
✔ in251
✔ in252
✔ in253
✔ in260
✔ in261
✔ in262
✔ in263
✔ in270
✔ in271
✔ in272
✔ in273
✔ in280
✔ in281
✔ in282
✔ in283
✔ in290
✔ in291
✔ in292
✔ in293
✔ in300
✔ in301
✔ in302
✘ Não encontrado: 30_3
✘ Não encontrado: 21_0
✘ Não encontrado: 21_1
✔ kn212
✔ kn213
✔ kn220
✔ kn221
✔ kn222
✔ kn223
✔ kn230
✔ kn231
✔ kn232
✔ kn233
✔ kn240
✔ kn241
✔ kn242
✔ kn243
✔ kn250
✔ kn251
✔ kn252
✔ kn253
✔ kn260
✔ kn261
✔ kn262
✔ kn263
✔ kn270
✔ kn271
✔ kn272
✔ kn273
✔ kn280
✔ kn281
✔ kn282
✔ kn283

In [10]:
# tratar manualmente a versão 30_3
versao = "30_3"

pasta = base_path / "onet" / f"db_{versao}_excel" / f"db_{versao}_excel"

arquivo_transferable = pasta / "Transferable Skills.xlsx"
arquivo_essential = pasta / "Essential Skills.xlsx"

try:
    # lê os dois arquivos
    df_transferable = pd.read_excel(arquivo_transferable)
    df_essential = pd.read_excel(arquivo_essential)

    # concatena verticalmente
    df_sk303 = pd.concat(
        [df_transferable, df_essential],
        ignore_index=True
    )

    # adiciona ao dicionário principal
    skills_dfs["sk303"] = df_sk303

    print("✔ sk303 criado com sucesso")

except FileNotFoundError as e:
    print(f"✘ Arquivo não encontrado: {e}")

✔ sk303 criado com sucesso


In [23]:
def filtrar_lv(dfs):

    novas_tabelas = {}

    colunas = [
        "O*NET-SOC Code",
        "Title",
        "Element ID",
        "Element Name",
        "Date",
        "Scale ID",
        "Scale Name",
        "Data Value"
    ]

    # mapeia prefixos -> tipo
    tipos = {
        "ab": "Abilities",
        "in": "Interests",
        "kn": "Knowledge",
        "sk": "Skills",
        "wa": "Work Activities",
        "ws": "Work Styles",
        "wv": "Work Values"
    }

    # mapeamento trimestre
    quarter_map = {
        "0": "Q3",
        "1": "Q4",
        "2": "Q1",
        "3": "Q2"
    }

    # referência:
    # 21.2 = 2017 Q1
    base_major = 21
    base_year = 2017

    for nome, df in dfs.items():

        # -------------------------
        # FILTRO
        # -------------------------

        if "Recommend Suppress" in df.columns:
            df_lv = df[df["Recommend Suppress"] != "Y"].copy()
        else:
            df_lv = df.copy()

        # mantém apenas colunas desejadas
        df_lv = df_lv[colunas].copy()

        # -------------------------
        # TYPE
        # -------------------------

        prefixo = nome[:2]
        tipo = tipos.get(prefixo, "Unknown")

        df_lv["Type"] = tipo

        # -------------------------
        # VERSION
        # exemplos:
        # wa291 -> 29.1
        # sk262 -> 26.2
        # -------------------------

        codigo_versao = nome[2:]  # ex: "291"

        major = codigo_versao[:2]   # "29"
        minor = codigo_versao[2]    # "1"

        major_int = int(major)

        version = f"{major}.{minor}"

        df_lv["Version"] = version

        # -------------------------
        # VERSION YEAR
        #
        # 21.2 = 2017 Q1
        #
        # .0 -> Q3 ano anterior
        # .1 -> Q4 ano anterior
        # .2 -> Q1 ano corrente
        # .3 -> Q2 ano corrente
        # -------------------------

        # ano principal da série
        series_year = base_year + (major_int - base_major)

        # .0 e .1 pertencem ao ano anterior
        if minor in ["0", "1"]:
            version_year = series_year - 1
        else:
            version_year = series_year

        df_lv["Version Year"] = version_year

        # -------------------------
        # QUARTER
        # -------------------------

        df_lv["Quarter"] = quarter_map.get(minor)

        # -------------------------
        # DATE
        # -------------------------

        datas = pd.to_datetime(
            df_lv["Date"],
            format="%m/%Y",
            errors="coerce"
        )

        # todas as datas anteriores a 2017 viram 07/2016
        datas = datas.mask(
            datas.dt.year < 2017,
            pd.Timestamp("2016-07-01")
        )

        # YEAR harmonizado
        df_lv["Year"] = datas.dt.year

        # opcional:
        # salvar Date harmonizada
        # df_lv["Date"] = datas.dt.strftime("%m/%Y")

        novas_tabelas[nome] = df_lv

        print(f"✔ {nome}")

    return novas_tabelas

In [24]:
# aplicar aos três conjuntos
abilities_lv = filtrar_lv(abilities_dfs)
# interests_lv = filtrar_lv(interests_dfs)
knowledge_lv = filtrar_lv(knowledge_dfs)
skills_lv = filtrar_lv(skills_dfs)
activities_lv = filtrar_lv(activities_dfs)
# styles_lv = filtrar_lv(styles_dfs)
# values_lv = filtrar_lv(values_dfs)

✔ ab212
✔ ab213
✔ ab220
✔ ab221
✔ ab222
✔ ab223
✔ ab230
✔ ab231
✔ ab232
✔ ab233
✔ ab240
✔ ab241
✔ ab242
✔ ab243
✔ ab250
✔ ab251
✔ ab252
✔ ab253
✔ ab260
✔ ab261
✔ ab262
✔ ab263
✔ ab270
✔ ab271
✔ ab272
✔ ab273
✔ ab280
✔ ab281
✔ ab282
✔ ab283
✔ ab290
✔ ab291
✔ ab292
✔ ab293
✔ ab300
✔ ab301
✔ ab302
✔ ab303
✔ kn212
✔ kn213
✔ kn220
✔ kn221
✔ kn222
✔ kn223
✔ kn230
✔ kn231
✔ kn232
✔ kn233
✔ kn240
✔ kn241
✔ kn242
✔ kn243
✔ kn250
✔ kn251
✔ kn252
✔ kn253
✔ kn260
✔ kn261
✔ kn262
✔ kn263
✔ kn270
✔ kn271
✔ kn272
✔ kn273
✔ kn280
✔ kn281
✔ kn282
✔ kn283
✔ kn290
✔ kn291
✔ kn292
✔ kn293
✔ kn300
✔ kn301
✔ kn302
✔ kn303
✔ sk212
✔ sk213
✔ sk220
✔ sk221
✔ sk222
✔ sk223
✔ sk230
✔ sk231
✔ sk232
✔ sk233
✔ sk240
✔ sk241
✔ sk242
✔ sk243
✔ sk250
✔ sk251
✔ sk252
✔ sk253
✔ sk260
✔ sk261
✔ sk262
✔ sk263
✔ sk270
✔ sk271
✔ sk272
✔ sk273
✔ sk280
✔ sk281
✔ sk282
✔ sk283
✔ sk290
✔ sk291
✔ sk292
✔ sk293
✔ sk300
✔ sk301
✔ sk302
✔ sk303
✔ wa212
✔ wa213
✔ wa220
✔ wa221
✔ wa222
✔ wa223
✔ wa230
✔ wa231
✔ wa232
✔ wa233
✔ wa240


In [17]:
va = values_lv['wv302']
va['YEAR'].unique()

array([2016, 2020])

In [26]:
display(knowledge_lv)

{'kn212':       O*NET-SOC Code                              Title Element ID  \
 0         11-1011.00                   Chief Executives    2.C.1.a   
 1         11-1011.00                   Chief Executives    2.C.1.a   
 2         11-1011.00                   Chief Executives    2.C.1.b   
 3         11-1011.00                   Chief Executives    2.C.1.b   
 4         11-1011.00                   Chief Executives    2.C.1.c   
 ...              ...                                ...        ...   
 63616     53-7121.00  Tank Car, Truck, and Ship Loaders    2.C.8.b   
 63618     53-7121.00  Tank Car, Truck, and Ship Loaders    2.C.9.a   
 63620     53-7121.00  Tank Car, Truck, and Ship Loaders    2.C.9.b   
 63622     53-7121.00  Tank Car, Truck, and Ship Loaders     2.C.10   
 63623     53-7121.00  Tank Car, Truck, and Ship Loaders     2.C.10   
 
                         Element Name     Date Scale ID  Scale Name  \
 0      Administration and Management  07/2014       IM  Importanc

In [18]:
sk = skills_lv['sk302']
sk['YEAR'].unique()

array([2023, 2021, 2018, 2016, 2025, 2017, 2024, 2022, 2020, 2019])

In [19]:
wa = activities_lv['wa302']
wa['YEAR'].unique()

array([2023, 2021, 2018, 2016, 2025, 2017, 2024, 2022, 2020, 2019])

In [27]:
# junta todos os dicionários em uma lista
todos_dfs = [
    abilities_lv,
    # interests_lv,
    knowledge_lv,
    skills_lv,
    activities_lv
    # styles_lv,
    # values_lv
]

# concatena todas as tabelas
df_onet = pd.concat(

    # pega todos os dataframes de todos os dicionários
    [df for grupo in todos_dfs for df in grupo.values()],

    ignore_index=True
)

# remove duplicatas completas
df_onet = df_onet.drop_duplicates().reset_index(drop=True)

print(f"Linhas finais: {len(df_onet):,}")
print(f"Duplicatas removidas: "
      f"{pd.concat([df for grupo in todos_dfs for df in grupo.values()]).shape[0] - len(df_onet):,}")

Linhas finais: 10,929,296
Duplicatas removidas: 0


In [32]:
# df_onet.to_excel("df_onet3.xlsx", index=False)
chunk_size = 500_000

for i in range(0, len(df_onet), chunk_size):

    chunk = df_onet.iloc[i:i + chunk_size]

    chunk.to_csv(
        "df_onet4.csv",
        mode="a",
        index=False,
        header=(i == 0)
    )

    print(f"✔ chunk {i}")

✔ chunk 0
✔ chunk 500000
✔ chunk 1000000
✔ chunk 1500000
✔ chunk 2000000
✔ chunk 2500000
✔ chunk 3000000
✔ chunk 3500000
✔ chunk 4000000
✔ chunk 4500000
✔ chunk 5000000
✔ chunk 5500000
✔ chunk 6000000
✔ chunk 6500000
✔ chunk 7000000
✔ chunk 7500000
✔ chunk 8000000
✔ chunk 8500000
✔ chunk 9000000
✔ chunk 9500000
✔ chunk 10000000
✔ chunk 10500000


In [10]:
# concatena todas as tabelas de occupation_dfs_merge
df_occupations = pd.concat(
    occupation_dfs_merge.values(),
    ignore_index=True
)

# remove duplicatas completas
df_occupations = (
    df_occupations
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Linhas finais: {len(df_occupations):,}")

Linhas finais: 7,359


In [11]:
df_occupations.to_excel("df_occ.xlsx", index=False)

In [33]:
way = Path(r'C:\Users\Maxwell\Documents\mestrado\dados\oesm\oesm16nat\oesm16nat\national_M2016_dl.xlsx')
oesm2016 = pd.read_excel(way)

In [49]:
# cria cópia da tabela base
od211 = occupation_dfs["od212"].copy()

# padroniza códigos O*NET -> remove .00
od211["OCC_CODE"] = (
    od211["O*NET-SOC Code"]
    .astype(str)
    .str.replace(r"\.00$", "", regex=True)
)

# colunas desejadas da OESM
oesm_cols = [
    "OCC_CODE",
    "OCC_TITLE",
    "TOT_EMP",
    "H_MEAN",
    "A_MEAN"
]

# merge
od211 = od211.merge(
    oesm2016[oesm_cols],
    on="OCC_CODE",
    how="left"
)

# mantém apenas matches
od211 = od211[od211["OCC_TITLE"].notna()].copy()

# cria YEAR
od211["YEAR"] = 2016

# cria MATCH_OESM
od211["MATCH_OESM"] = True

# remove coluna auxiliar
od211 = od211.drop(columns=["OCC_CODE"])

# adiciona ao dicionário
occupation_dfs_merge["od211"] = od211

print("✔ od211 criado")
print(od211.shape)

✔ od211 criado
(818, 9)


In [50]:
occupation_dfs_merge["od211"]

,O*NET-SOC Code,Title,Description,OCC_TITLE,TOT_EMP,H_MEAN,A_MEAN,YEAR,MATCH_OESM
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...,Chief Executives,223260.0,93.44,194350,2016,True
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",General and Operations Managers,2188870.0,58.7,122090,2016,True
3,11-1031.00,Legislators,"Develop, introduce or enact laws and statutes ...",Legislators,53670.0,*,44820,2016,True
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",Advertising and Promotions Managers,28860.0,56.64,117810,2016,True
6,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies...",Marketing Managers,205900.0,69.3,144140,2016,True
...,...,...,...,...,...,...,...,...,...
1085,53-7073.00,Wellhead Pumpers,Operate power pumps and auxiliary equipment to...,Wellhead Pumpers,11610.0,24.39,50730,2016,True
1086,53-7081.00,Refuse and Recyclable Material Collectors,Collect and dump refuse or recyclable material...,Refuse and Recyclable Material Collectors,114680.0,18.12,37690,2016,True
1087,53-7111.00,Mine Shuttle Car Operators,Operate diesel or electric-powered shuttle car...,Mine Shuttle Car Operators,1590.0,27.1,56370,2016,True
1088,53-7121.00,"Tank Car, Truck, and Ship Loaders","Load and unload chemicals and bulk solids, suc...","Tank Car, Truck, and Ship Loaders",10920.0,19.04,39590,2016,True


In [29]:
df_onet

,O*NET-SOC Code,Title,Element ID,Element Name,Date,Scale ID,Scale Name,Data Value,Type,Version,Version Year,Quarter,Year
0,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,07/2014,IM,Importance,4.50,Abilities,21.2,2017,Q1,2016
1,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,07/2014,LV,Level,4.88,Abilities,21.2,2017,Q1,2016
2,11-1011.00,Chief Executives,1.A.1.a.2,Written Comprehension,07/2014,IM,Importance,4.25,Abilities,21.2,2017,Q1,2016
3,11-1011.00,Chief Executives,1.A.1.a.2,Written Comprehension,07/2014,LV,Level,4.62,Abilities,21.2,2017,Q1,2016
4,11-1011.00,Chief Executives,1.A.1.a.3,Oral Expression,07/2014,IM,Importance,4.38,Abilities,21.2,2017,Q1,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10929291,53-7121.00,"Tank Car, Truck, and Ship Loaders",4.A.4.c.1,Performing Administrative Activities,08/2019,LV,Level,2.27,Work Activities,30.3,2026,Q2,2019
10929292,53-7121.00,"Tank Car, Truck, and Ship Loaders",4.A.4.c.2,Staffing Organizational Units,08/2019,IM,Importance,1.93,Work Activities,30.3,2026,Q2,2019
10929293,53-7121.00,"Tank Car, Truck, and Ship Loaders",4.A.4.c.2,Staffing Organizational Units,08/2019,LV,Level,1.60,Work Activities,30.3,2026,Q2,2019
10929294,53-7121.00,"Tank Car, Truck, and Ship Loaders",4.A.4.c.3,Monitoring and Controlling Resources,08/2019,IM,Importance,2.56,Work Activities,30.3,2026,Q2,2019
